# Custom proatom states and expert pipelines

The public `atomref-proatoms generate` CLI intentionally uses only curated
`neutral` and `stockholder` state policies. This notebook shows the expert
Python route for custom configurations, custom multiplicities, and project-specific
pipelines.

The goal is not to hide the chemistry behind helper functions. A custom proatom
state should be written down plainly: element, charge, electron count, spin
multiplicity, compact configuration, and alpha/beta electron counts in each
angular-momentum block.

The notebook is executable as written. The actual SCF calculation cell is guarded
by `RUN_OPTIONAL_CUSTOM_SCF = False` so docs can be built without PySCF or
expensive quantum-chemistry work. Set it to `True` locally when you want to run
the tiny neutral-H smoke calculation and inspect the resulting profile, `.rad`,
and `.wfn` files.


## 1. Inspect curated states first

Before inventing a custom state, inspect the curated state table. For neutral
atoms and common cations, curated records are usually the safest starting point.
For anions and transition metals, state selection is a scientific choice, not a
numerical afterthought.


In [1]:
from atomref_proatoms import select_packaged_states

selection = select_packaged_states(
    elements=("C", "Ni"),
    policy="stockholder",
    charges=None,
)

for state in selection.states:
    print(
        state.state_id,
        state.symbol,
        state.charge,
        state.multiplicity,
        state.record["configuration"],
    )

C_qm3_mult2_formal C -3 2 1s2 2s2 2p5
C_qm2_mult3_formal C -2 3 1s2 2s2 2p4
C_qm1_mult4_ning2022 C -1 4 [He] 2s2 2p3
C_q0_mult3_nist C 0 3 1s2 2s2 2p2
C_qp1_mult2_nist C 1 2 1s2 2s2 2p1
C_qp2_mult1_nist C 2 1 1s2 2s2
C_qp3_mult2_nist C 3 2 1s2 2s1
Ni_qm1_mult2_ning2022 Ni -1 2 [Ar] 3d9 4s2
Ni_q0_mult3_nist Ni 0 3 [Ar] 3d8 4s2
Ni_qp1_mult2_nist Ni 1 2 [Ar] 3d9
Ni_qp2_mult3_nist Ni 2 3 [Ar] 3d8
Ni_qp3_mult4_nist Ni 3 4 [Ar] 3d7


## 2. Count a concrete custom state by hand

The spherical atom model needs total alpha and beta electron counts for each
angular-momentum manifold `l`, aggregated over radial shells with the same
angular momentum:

```text
l = 0  s
l = 1  p
l = 2  d
l = 3  f
```

As a small diagnostic example, use a carbon anion with configuration
`[He] 2s2 2p3` and multiplicity 4. This is a teaching record, not an
experimental claim.

The charge is `-1`, so carbon has seven electrons. Multiplicity 4 means
`2S = multiplicity - 1 = 3`, therefore:

```text
N_alpha = (7 + 3)/2 = 5
N_beta  = (7 - 3)/2 = 2
```

The closed `1s2 2s2` part contributes four s electrons: two alpha and two beta.
The high-spin `2p3` part contributes three alpha p electrons. The aggregate
counts are therefore:

```text
alpha_l_counts = {0: 2, 1: 3}
beta_l_counts  = {0: 2, 1: 0}
```

This explicit counting is often clearer than a general-purpose parser for custom
states.


In [2]:
def alpha_beta_total_counts(electron_count: int, multiplicity: int) -> tuple[int, int]:
    """Return total alpha and beta electron counts from N and multiplicity."""

    spin_2s = multiplicity - 1
    if (electron_count + spin_2s) % 2:
        raise ValueError("electron_count and multiplicity have inconsistent parity")
    alpha = (electron_count + spin_2s) // 2
    beta = (electron_count - spin_2s) // 2
    if alpha < beta or beta < 0:
        raise ValueError("invalid alpha/beta counts")
    return alpha, beta


# Carbon anion diagnostic state: [He] 2s2 2p3, multiplicity 4.
electron_count = 7
multiplicity = 4
total_alpha, total_beta = alpha_beta_total_counts(electron_count, multiplicity)

alpha_l_counts = {0: 2.0, 1: 3.0}
beta_l_counts = {0: 2.0, 1: 0.0}

assert sum(alpha_l_counts.values()) == total_alpha
assert sum(beta_l_counts.values()) == total_beta
assert sum(alpha_l_counts.values()) - sum(beta_l_counts.values()) == multiplicity - 1

print("total alpha/beta =", total_alpha, total_beta)
print("alpha_l_counts =", alpha_l_counts)
print("beta_l_counts  =", beta_l_counts)


total alpha/beta = 5 2
alpha_l_counts = {0: 2.0, 1: 3.0}
beta_l_counts  = {0: 2.0, 1: 0.0}


## 3. Build an `AtomState`-like custom record

The record below is a custom diagnostic state. It uses the same schema constants
as packaged v2 state records, but it is not part of the packaged state table and
should not be interpreted as an accepted isolated-ion reference.

For production data, prefer curated records. For local diagnostics, make the
state record honest and explicit.


In [3]:
from atomref_proatoms import AtomState, validate_atom_state

custom_c_anion = {
    "schema_version": "atomref.proatoms.state.v2",
    "state_id": "C_qm1_mult4_custom_s2p3",
    "symbol": "C",
    "z": 6,
    "charge": -1,
    "electron_count": 7,
    "spin_2s": 3,
    "multiplicity": 4,
    "configuration": "[He] 2s2 2p3",
    "state_role": "custom_diagnostic",
    "state_category": "formal_anion_reference",
    "curation_status": "custom_not_packaged",
    "spin_model": "curated_ground_multiplicity",
    "spin_variant": "curated_multiplicity",
    "occupation_policy": "spherical_l_counts_from_curated_multiplicity_v2",
    "alpha_l_counts": {str(key): value for key, value in alpha_l_counts.items()},
    "beta_l_counts": {str(key): value for key, value in beta_l_counts.items()},
    "notes": [
        "Notebook custom diagnostic state; not part of the packaged state table.",
        "Not a claim of a stable isolated atomic anion.",
    ],
}

errors = validate_atom_state(custom_c_anion)
if errors:
    raise ValueError(errors)

custom_state = AtomState(custom_c_anion)
print(custom_state.state_id)
print("alpha:", custom_state.alpha_l_counts)
print("beta: ", custom_state.beta_l_counts)


C_qm1_mult4_custom_s2p3
alpha: {0: 2.0, 1: 3.0}
beta:  {0: 2.0, 1: 0.0}


## 4. A tiny neutral custom state for optional execution

The carbon-anion record above is useful for explaining custom state bookkeeping,
but it is not a good default notebook calculation. For a lightweight smoke test
that can also export a `.wfn`, use a neutral one-electron H state.


In [4]:
custom_h = AtomState(
    {
        "schema_version": "atomref.proatoms.state.v2",
        "state_id": "H_q0_mult2_custom_1s1",
        "symbol": "H",
        "z": 1,
        "charge": 0,
        "electron_count": 1,
        "spin_2s": 1,
        "multiplicity": 2,
        "configuration": "1s1",
        "state_role": "custom_diagnostic",
        "state_category": "nist_reference",
        "curation_status": "custom_not_packaged",
        "spin_model": "curated_ground_multiplicity",
        "spin_variant": "curated_multiplicity",
        "occupation_policy": "spherical_l_counts_from_curated_multiplicity_v2",
        "alpha_l_counts": {"0": 1.0},
        "beta_l_counts": {"0": 0.0},
        "notes": ["Tiny neutral custom-state smoke-test record."],
    }
)

errors = validate_atom_state(custom_h.record)
if errors:
    raise ValueError(errors)

print(custom_h.state_id, custom_h.alpha_l_counts, custom_h.beta_l_counts)


H_q0_mult2_custom_1s1 {0: 1.0} {0: 0.0}


## 5. Basis and method choices

The CLI accepts PySCF basis names, BSE basis names, and local NWChem files. In
Python, use the same principle: resolve the basis explicitly and record the
source. The generator utilities added for the CLI are also useful for checks and
metadata in custom scripts.


In [5]:
from pathlib import Path

from atomref_proatoms import (
    check_basis_source,
    check_method_with_pyscf,
    parse_basis_spec,
    parse_method,
    parse_relativity,
)

method = parse_method("PBE0")
relativity = parse_relativity("none")
pyscf_basis = parse_basis_spec(basis="pyscf:sto-3g", basis_file=None, basis_name=None)

print(method)
print(relativity)
print(check_method_with_pyscf(method).as_dict())
print(check_basis_source(pyscf_basis, ("H",)).as_dict())

local_basis_candidates = [
    Path("examples/02_cli_stockholder_local_basis/input/dyall-v2z-ni-pd-pt.nw"),
    Path("../02_cli_stockholder_local_basis/input/dyall-v2z-ni-pd-pt.nw"),
]
local_basis_path = next((path for path in local_basis_candidates if path.exists()), None)
if local_basis_path is not None:
    local_basis = parse_basis_spec(
        basis_file=local_basis_path,
        basis_name="dyall-v2z-ni-pd-pt",
    )
    print(check_basis_source(local_basis, ("Ni", "Pd")).as_dict())
else:
    print("Local dyall-v2z Ni/Pd/Pt example basis file was not found from this working directory.")

MethodSpec(input_method='PBE0', method_kind='dft', xc='PBE0', scf_type='UKS')
RelativitySpec(input_relativity='none', relativity='none', engine_label='none')
{'status': 'ok', 'message': "PySCF accepted XC string 'PBE0'.", 'pyscf_version': '2.13.1'}
{'basis_source': {'source': 'pyscf', 'name': 'sto-3g', 'label': 'sto-3g', 'path': None, 'original': 'pyscf:sto-3g', 'basis_key': 'pyscf:sto-3g'}, 'requested_symbols': ['H'], 'status': 'ok', 'full_electron_status': 'no_ecp_detected', 'coverage_checked': True, 'covered_symbols': ['H'], 'missing_symbols': [], 'ecp_detected': False, 'ecp_symbols': [], 'check_method': 'pyscf_basis_load', 'messages': ['PySCF version: 2.13.1'], 'errors': [], 'warnings': [], 'basis_sha256': None}
{'basis_source': {'source': 'file', 'name': 'dyall-v2z-ni-pd-pt', 'label': 'dyall-v2z-ni-pd-pt', 'path': '../02_cli_stockholder_local_basis/input/dyall-v2z-ni-pd-pt.nw', 'original': '../02_cli_stockholder_local_basis/input/dyall-v2z-ni-pd-pt.nw', 'basis_key': 'file:dyall-v2

## 6. Optional custom SCF calculation

The cell below is real executable code, but the SCF call is disabled by default so
that this notebook can be executed in documentation environments without PySCF.
Set `RUN_OPTIONAL_CUSTOM_SCF = True` locally to run a tiny neutral-H calculation
and write profile, radii, `.rad`, and `.wfn` outputs under `output/notebook_h/`.


In [6]:
import numpy as np

from atomref_proatoms import (
    apply_x2c_if_requested,
    atom_wfn_filename,
    configure_dft_grid,
    density_profile_from_mf,
    derived_radii,
    evaluate_scf_radial_density,
    interpolate_density,
    log_radial_grid,
    make_spherical_uks,
    multiwfn_rad_filename,
    validate_spherical_ao_layout,
    write_atomref_spherical_wfn,
    write_multiwfn_rad_file,
    write_scf_npz,
    write_wide_profiles_csv,
)

RUN_OPTIONAL_CUSTOM_SCF = False


def run_tiny_custom_h_pipeline(output_dir: Path) -> dict[str, Path]:
    """Run a tiny custom neutral-H spherical UKS pipeline and write artifacts."""

    from pyscf import gto  # type: ignore[import-not-found]

    output_dir.mkdir(parents=True, exist_ok=True)
    mol = gto.Mole()
    mol.atom = "H 0 0 0"
    mol.basis = {"H": "sto-3g"}
    mol.charge = custom_h.charge
    mol.spin = custom_h.spin_2s
    mol.unit = "Bohr"
    mol.cart = False
    mol.symmetry = False
    mol.verbose = 0
    mol.build()
    validate_spherical_ao_layout(mol)

    mf = make_spherical_uks(
        mol,
        xc="PBE0",
        alpha_l_counts=custom_h.alpha_l_counts,
        beta_l_counts=custom_h.beta_l_counts,
    )
    mf.conv_tol = 1e-9
    mf.max_cycle = 100
    configure_dft_grid(mf, level=0, prune=None)
    mf = apply_x2c_if_requested(mf, use_x2c=False)
    mf.kernel()
    if not mf.converged:
        raise RuntimeError("tiny custom H SCF did not converge")

    write_scf_npz(output_dir / "scf.npz", mf)

    r_grid = log_radial_grid(1.0e-5, 20.0, 200)
    profile = density_profile_from_mf(mf, r_grid=r_grid, compute_qa=False)
    write_wide_profiles_csv(
        output_dir / "profiles.csv",
        r_bohr=profile["r_bohr"],
        densities_by_state_id={custom_h.state_id: profile["rho_e_bohr3"]},
    )

    radii = derived_radii(profile["r_bohr"], profile["rho_e_bohr3"])
    (output_dir / "radii.json").write_text(
        __import__("json").dumps(radii, indent=2, sort_keys=True),
        encoding="utf-8",
    )

    dm_alpha, dm_beta = mf.make_rdm1()
    r_rad, rho_rad = evaluate_scf_radial_density(
        mol,
        np.asarray(dm_alpha) + np.asarray(dm_beta),
    )
    rad_path = output_dir / multiwfn_rad_filename(custom_h.symbol, custom_h.charge)
    write_multiwfn_rad_file(rad_path, r_rad, rho_rad)

    wfn_path = output_dir / atom_wfn_filename(custom_h.symbol)
    write_atomref_spherical_wfn(wfn_path, custom_h, mf)

    sample_r = np.array([0.2, 0.5, 1.0, 2.0])
    sample_rho = interpolate_density(profile["r_bohr"], profile["rho_e_bohr3"], sample_r)
    for r_value, rho_value in zip(sample_r, sample_rho, strict=True):
        print(f"rho({r_value:.2f} bohr) = {rho_value:.6e} e/bohr^3")

    return {
        "scf_npz": output_dir / "scf.npz",
        "profiles_csv": output_dir / "profiles.csv",
        "radii_json": output_dir / "radii.json",
        "rad": rad_path,
        "wfn": wfn_path,
    }


if RUN_OPTIONAL_CUSTOM_SCF:
    written = run_tiny_custom_h_pipeline(Path("output/notebook_h"))
    for label, path in written.items():
        print(f"{label}: {path}")
else:
    print("Optional SCF skipped. Set RUN_OPTIONAL_CUSTOM_SCF = True to run it locally.")

Optional SCF skipped. Set RUN_OPTIONAL_CUSTOM_SCF = True to run it locally.


## 7. Profile interpolation without running SCF

Use `atomref_proatoms.interpolate_density` when you need `rho(r)` at custom
radii. Units are explicit: radii are in bohr and densities are in electron/bohr^3.


In [7]:
r_bohr = np.array([0.1, 0.2, 0.5, 1.0, 2.0])
rho = np.array([100.0, 40.0, 5.0, 0.5, 0.01])
query = np.array([0.15, 0.75, 1.5])

print(interpolate_density(r_bohr, rho, query))

[5.85086419e+01 1.30019204e+00 5.07149804e-02]


## 8. Recommendations for custom pipelines

- Use curated/NIST/literature-backed states whenever possible.
- Treat transition-metal, lanthanide, actinide, and anion scans as diagnostics unless the state has a clear source.
- Do not silently compare all-electron and ECP/effective-valence densities.
- Keep method, basis, relativistic convention, state definition, and package version in every manifest or filename namespace.
- Prefer small smoke tests before launching a large state grid.
- Make alpha/beta `l` counts explicit rather than relying on implicit Aufbau/Hund-rule reconstruction.
